In [8]:
!pip install streamlit -q

In [9]:
%%writefile app.py
import streamlit as st
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.neural_network import MLPClassifier
from sklearn.preprocessing import StandardScaler

@st.cache_data
def load_data():
    # Data diambil via raw URL GitHub agar dapat langsung diakses sistem tanpa perlu konfigurasi API Key Kaggle.
    url = "https://raw.githubusercontent.com/jbrownlee/Datasets/master/pima-indians-diabetes.data.csv"
    columns = ['Pregnancies', 'Glucose', 'BloodPressure', 'SkinThickness', 'Insulin', 'BMI', 'DiabetesPedigreeFunction', 'Age', 'Outcome']
    # Memuat data menjadi bentuk DataFrame.
    df = pd.read_csv(url, names=columns)
    return df

@st.cache_data
def train_model(df):
    features = ["Glucose", "BMI", "Age", "BloodPressure"]
    X = df[features]
    y = df["Outcome"]

    # Membagi data menjadi 80% data latih (training) dan 20% data uji (testing).
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

    # Standarisasi Data (Preprocessing).
    # Angka glukosa (ratusan) dan BMI (puluhan).
    scaler = StandardScaler()
    X_train_scaled = scaler.fit_transform(X_train)
    X_test_scaled = scaler.transform(X_test)

    # Membangun Arsitektur JST (Multi-Layer Perceptron).
    model = MLPClassifier(hidden_layer_sizes=(8, 4), activation='relu', max_iter=500, random_state=42)
    # Proses iterasi perhitungan bobot dan bias secara otomatis (Forward & Backward Propagation)
    model.fit(X_train_scaled, y_train)

    # Mengukur seberapa baik prediksi model pada 20% data uji
    accuracy = model.score(X_test_scaled, y_test)
    return model, scaler, accuracy

def main():
    st.set_page_config(page_title="Klasifikasi Diabetes JST", layout="centered")
    st.title("Aplikasi Prediksi Diabetes dengan JST (ANN)")
    st.write("Geser slider di bawah ini untuk memprediksi potensi pasien terkena diabetes (Ya / Tidak):")

    df = load_data()
    model, scaler, accuracy = train_model(df)

    glucose = st.slider("Kadar Glukosa", float(df.Glucose.min()), float(df.Glucose.max()), float(df.Glucose.mean()))
    bmi = st.slider("BMI (Indeks Massa Tubuh)", float(df.BMI.min()), float(df.BMI.max()), float(df.BMI.mean()))
    age = st.slider("Usia", float(df.Age.min()), float(df.Age.max()), float(df.Age.mean()))
    bp = st.slider("Tekanan Darah", float(df.BloodPressure.min()), float(df.BloodPressure.max()), float(df.BloodPressure.mean()))

    if st.button("Prediksi Sekarang"):
        # Fungsi untuk mengambil data dari antarmuka (UI) sebagai input data baru
        input_data = np.array([[glucose, bmi, age, bp]])

        # Fungsi untuk scaling pada input baru
        input_scaled = scaler.transform(input_data)

        # Fungsi predict() melakukan perhitungan maju (forward) melewati bobot JST
        prediction = model.predict(input_scaled)[0]

        keputusan = "Ya (Positif Diabetes)" if prediction == 1 else "Tidak (Negatif Diabetes)"
        st.success(f"🩺 Prediksi Model: **{keputusan}**")
        st.info(f"Akurasi JST pada data uji: {accuracy*100:.2f}%")

    if st.checkbox("Tampilkan Dataset"):
        st.dataframe(df)

if __name__ == "__main__":
    main()

Overwriting app.py


In [11]:
# Mengunduh modul Cloudflare
!wget -q -O cloudflared-linux-amd64 https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64
!chmod +x cloudflared-linux-amd64

# Menjalankan Streamlit secara bersamaan dengan Cloudflare Tunnel
!streamlit run app.py & ./cloudflared-linux-amd64 tunnel --url http://localhost:8501

2026-05-12T05:50:16Z INF Thank you for trying Cloudflare Tunnel. Doing so, without a Cloudflare account, is a quick way to experiment and try it out. However, be aware that these account-less Tunnels have no uptime guarantee, are subject to the Cloudflare Online Services Terms of Use (https://www.cloudflare.com/website-terms/), and Cloudflare reserves the right to investigate your use of Tunnels for violations of such terms. If you intend to use Tunnels in production you should use a pre-created named tunnel by following: https://developers.cloudflare.com/cloudflare-one/connections/connect-apps
2026-05-12T05:50:16Z INF Requesting new quick Tunnel on trycloudflare.com...


2026-05-12 05:50:18.320 Uvicorn server started on 0.0.0.0:8501

  You can now view your Streamlit app in your browser.

  Local URL: http://localhost:8501
  Network URL: http://172.28.0.12:8501
  External URL: http://34.24.59.113:8501

2026-05-12T05:50:20Z INF +---------------------------------------------------------